# Welcome to DoIT AI Cluster JupyterLab

This notebook will help you get started with the GPU cluster.

## Your Storage

- **`/home/jovyan/work/`** - Your personal 30GB workspace (writable)
- **`/models/`** - Shared model repository (read-only)

## Available Models

Pre-downloaded models in `/models/.cache/huggingface/`:

- Qwen/Qwen2.5-7B-Instruct (~15GB)
- Qwen/Qwen2.5-14B-Instruct (~29GB)
- Qwen/Qwen2.5-32B-Instruct (~66GB)
- Qwen/Qwen2.5-72B-Instruct (~145GB)
- Qwen/Qwen2.5-110B-A22B-Instruct (~110GB)
- **Qwen/Qwen3.5-35B-A3B** (~67GB) - Latest! Released Feb 24, 2026

Plus embedding models and more!

In [1]:
import subprocess

print(subprocess.run(
    ["bash", "-lc", "ls -1 /models/.cache/huggingface | grep models--"],
    capture_output=True,
    text=True
).stdout)

models--Qwen--Qwen1.5-110B-Chat
models--Qwen--Qwen2.5-14B-Instruct
models--Qwen--Qwen2.5-72B-Instruct
models--Qwen--Qwen2.5-7B-Instruct
models--Qwen--Qwen3-30B-A3B-Instruct-2507
models--Qwen--Qwen3-Next-80B-A3B-Instruct
models--Qwen--Qwen3-Next-80B-A3B-Thinking-FP8
models--Qwen--Qwen3.5-35B-A3B
models--jinaai--jina-embeddings-v4



## Example 1: Check Your GPU

In [2]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.7.0a0+ecf3bae40a.nv25.02
CUDA available: True
CUDA version: 12.8
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU Memory: 51.3 GB


In [3]:
import os

os.environ["HF_HOME"] = "/home/jovyan/work/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/home/jovyan/work/.cache/huggingface/hub"
os.environ["TRANSFORMERS_CACHE"] = "/home/jovyan/work/.cache/huggingface/hub"
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)

## Example 2: Check Your Storage

In [4]:
import subprocess
import os

# Check your workspace size
print("=== Your Workspace ===")
result = subprocess.run(['df', '-h', '/home/jovyan/work'], capture_output=True, text=True)
print(result.stdout)

# Check current usage
result = subprocess.run(['du', '-sh', '/home/jovyan/work'], capture_output=True, text=True)
print(f"Current usage: {result.stdout.strip()}")

# Check shared models
print("\n=== Shared Models ===")
result = subprocess.run(['du', '-sh', '/models/.cache/huggingface/'], capture_output=True, text=True)
print(f"Total models: {result.stdout.strip()}")

=== Your Workspace ===
Filesystem                      Size  Used Avail Use% Mounted on
/dev/mapper/Volume00-runai_pvc  1.0T  781G  244G  77% /home/jovyan/work

Current usage: 56K	/home/jovyan/work

=== Shared Models ===
Total models: 744G	/models/.cache/huggingface/


In [5]:
import subprocess

print(subprocess.run(
    ["bash", "-lc", "ls -1 /models/.cache/huggingface | grep models--"],
    capture_output=True,
    text=True
).stdout)

models--Qwen--Qwen1.5-110B-Chat
models--Qwen--Qwen2.5-14B-Instruct
models--Qwen--Qwen2.5-72B-Instruct
models--Qwen--Qwen2.5-7B-Instruct
models--Qwen--Qwen3-30B-A3B-Instruct-2507
models--Qwen--Qwen3-Next-80B-A3B-Instruct
models--Qwen--Qwen3-Next-80B-A3B-Thinking-FP8
models--Qwen--Qwen3.5-35B-A3B
models--jinaai--jina-embeddings-v4



## Example 3: Load a Model from Shared Cache

This loads the latest Qwen3.5-35B-A3B model (Claude Sonnet 4.5 performance!)

In [6]:
import os

base = "/models/.cache/huggingface/models--Qwen--Qwen2.5-7B-Instruct/snapshots"
snapshots = sorted(os.listdir(base))

print("Snapshots:", snapshots)

LLM = os.path.join(base, snapshots[-1])
print("Using snapshot:", LLM)

Snapshots: ['a09a35458c702b33eeacc393d103063234e8bc28']
Using snapshot: /models/.cache/huggingface/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    LLM,
    local_files_only=True,
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    LLM,
    local_files_only=True,
    device_map="auto",
    dtype=torch.bfloat16,
    quantization_config=quant_config,
)

print("Model loaded")

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded


## Example 4: Generate Text

In [8]:
# Create a chat message
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "Explain what a GPU cluster is in simple terms."}
]

# Apply chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize
inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generate
print("Generating response...")
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

# Decode and print
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n" + "="*50)
print(response)
print("="*50)

Generating response...

system
You are a helpful AI assistant.
user
Explain what a GPU cluster is in simple terms.
assistant
Sure! Imagine you have a group of friends, and each one is really good at doing a specific task quickly—like one friend who can solve math problems fast, another who can remember faces, and so on. A GPU cluster is kind of like having a team of these super-fast workers all working together.

In simpler terms:

1. **GPUs (Graphics Processing Units)**: These are special types of computer chips that are very good at handling complex calculations and data processing tasks, especially those related to graphics or machine learning.

2. **Cluster**: This means a collection of multiple devices working together as a single system.

So, a **GPU cluster** is a group of computers (or servers) that each have powerful GPUs. These GPUs work together to perform massive amounts of calculations much faster than a single computer could. They're often used for tasks like training art

## Example 5: List Available Models

In [9]:
import os

models_dir = "/models/.cache/huggingface/"

print("Available models in shared cache:\n")

for item in sorted(os.listdir(models_dir)):
    if item.startswith("models--"):
        # Convert directory name to model name
        model_name = item.replace("models--", "").replace("--", "/")
        
        # Get size
        model_path = os.path.join(models_dir, item)
        result = subprocess.run(
            ['du', '-sh', model_path],
            capture_output=True,
            text=True
        )
        size = result.stdout.split()[0]
        
        print(f"  {model_name:<50} {size:>8}")

Available models in shared cache:

  Qwen/Qwen1.5-110B-Chat                                 208G
  Qwen/Qwen2.5-14B-Instruct                               28G
  Qwen/Qwen2.5-72B-Instruct                              136G
  Qwen/Qwen2.5-7B-Instruct                                15G
  Qwen/Qwen3-30B-A3B-Instruct-2507                        57G
  Qwen/Qwen3-Next-80B-A3B-Instruct                       152G
  Qwen/Qwen3-Next-80B-A3B-Thinking-FP8                    77G
  Qwen/Qwen3.5-35B-A3B                                    67G
  jinaai/jina-embeddings-v4                              7.1G


## Example 6: Save Your Work

**IMPORTANT:** Always save to `/home/jovyan/work/`

- `/models/` is READ-ONLY (shared with everyone)
- `/home/jovyan/work/` is YOUR writable 30GB space

In [10]:
import pandas as pd

# Create some data
data = {
    'model': ['Qwen2.5-7B', 'Qwen2.5-14B', 'Qwen3.5-35B'],
    'size_gb': [15, 29, 67],
    'released': ['2024', '2024', '2026']
}

df = pd.DataFrame(data)

# ✅ CORRECT - Save to your workspace
df.to_csv('/home/jovyan/work/my_models.csv', index=False)
print("✅ Saved to /home/jovyan/work/my_models.csv")

# ❌ WRONG - This will fail (read-only)
# df.to_csv('/models/my_models.csv', index=False)

✅ Saved to /home/jovyan/work/my_models.csv


## Getting Help

- **Documentation:** [Internal wiki or docs link]
- **Support:** Contact DoIT AI Team
- **Model requests:** Email [contact] to request new models

## Best Practices

1. **Use quantization** - Load models with `load_in_4bit=True` or `load_in_8bit=True` to save GPU memory
2. **Monitor GPU usage** - Use `nvidia-smi` in terminal to check GPU utilization
3. **Clean up** - Delete old checkpoints and temporary files from `/home/jovyan/work/`
4. **Share notebooks** - Save interesting notebooks to a shared location or Git repo
5. **Stop workspaces** - Stop your workspace when not in use to free up GPUs

Happy coding! 🚀